# 01 — Data Exploration (Phase 1)

Objective: profile the flow datasets (shape, dtypes, missing/inf, labels,
imbalance) and freeze the quality gates Q-01/Q-02 (DRD §A5).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import CATEGORY_COL, LABEL_COL
from src.data_loader import (FEATURE_NAMES, generate_shift_dataset,
                               generate_synthetic_dataset, label_mapping_report,
                               load_dataset)

RAW = ROOT / "data" / "raw"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

# Deterministic synthetic ingest (real CSVs can be dropped into data/raw instead).
generate_synthetic_dataset().to_csv(RAW / "synthetic_A.csv", index=False)
generate_shift_dataset().to_csv(RAW / "synthetic_B.csv", index=False)
A = load_dataset(RAW / "synthetic_A.csv")
B = load_dataset(RAW / "synthetic_B.csv")
print("A:", A.shape, " B:", B.shape)

In [ ]:
featsA = [c for c in A.columns if c in FEATURE_NAMES]
print("numeric features in A:", len(featsA))
print("missing rate:", round(float(A[featsA].isna().mean().mean()), 4))
print("inf rate:    ", round(float(np.isinf(A[featsA].to_numpy(dtype=float, na_value=np.nan)).mean()), 4))
print(A.dtypes.value_counts())

In [ ]:
counts = A[LABEL_COL].value_counts()
print(counts)
print("\nbenign share:", round((A[CATEGORY_COL] == "BENIGN").mean(), 3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_title("Class imbalance (synthetic_A)")
ax.set_ylabel("flows")
fig.tight_layout(); fig.savefig(FIG / "class_imbalance.png", dpi=110); plt.close(fig)
print("saved class_imbalance.png")

In [ ]:
prof = pd.DataFrame([
    {"dataset": n, "rows": len(d),
     "features": len([c for c in d.columns if c in FEATURE_NAMES]),
     "benign": int((d[CATEGORY_COL] == "BENIGN").sum()),
     "attacks": int((d[CATEGORY_COL] != "BENIGN").sum())}
    for n, d in (("synthetic_A", A), ("synthetic_B", B))])
prof.to_csv(TAB / "dataset_profile.csv", index=False)
label_mapping_report(A).to_csv(TAB / "label_map.csv", index=False)
prof

## Checkpoint — Phase 1 verdict

- Q-01: missing/inf rates are ~1% combined and explicitly handled downstream.
- Q-02: benign majority with 4 attack slices of ≥800 flows each — no slice is low-support.
- Q-03 (constant/duplicate check) runs in notebook 02 before PCA training.